# KL 散度手撕实现

## 1. 定义
$$D_{KL}(P\|Q)=\sum_i P_i \log\frac{P_i}{Q_i}=\sum_i P_i(\log P_i - \log Q_i)$$
- **非对称**：$D_{KL}(P\|Q)\ne D_{KL}(Q\|P)$，故"散度"非"距离"。
- **非负**：$D_{KL}\ge 0$，当且仅当 $P=Q$ 时为 0（Gibbs 不等式）。
- 与交叉熵关系：$H(P,Q)=H(P)+D_{KL}(P\|Q)$。

## 2. 输入约定
- 输入应是**概率分布**（非负、和为 1）。若拿到的是 logits，需先 softmax 归一化。
- **PyTorch `F.kl_div` 的坑**：其签名 `kl_div(input, target)` 中 `input=\log Q`、`target=P`，且默认 `reduction='mean'` 按**元素**取平均（不是按分布求和）。使用时通常要 `reduction='batchmean'` 才是真正的 KL。

## 3. 对称化：JS 散度
$$JSD(P,Q)=\tfrac12 D_{KL}(P\|M)+\tfrac12 D_{KL}(Q\|M),\ M=\tfrac12(P+Q)$$
有界 $\in[0,\log 2]$，对称且仍是合法散度。

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
# 构造两个概率分布（先 logits 再 softmax，保证合法）
logits_p = torch.randn(3, 5)
logits_q = torch.randn(3, 5)
P = F.softmax(logits_p, dim=1)
Q = F.softmax(logits_q, dim=1)

In [ ]:
# 手撕 KL(P||Q) = sum P * (log P - log Q)，逐行求和
def kl_divergence(P, Q, eps=1e-12):
    return (P * (torch.log(P + eps) - torch.log(Q + eps))).sum(dim=1)

kl_manual = kl_divergence(P, Q)
print('KL(P||Q):', kl_manual)
print('非负:', (kl_manual >= -1e-6).all().item())

In [ ]:
# 与 PyTorch F.kl_div 对比（注意约定：input=log Q, target=P, reduction=batchmean）
kl_torch = F.kl_div(torch.log(Q), P, reduction='batchmean')
print('F.kl_div(batchmean):', kl_torch.item())
print('manual mean        :', kl_manual.mean().item())
print('diff:', abs(kl_torch.item() - kl_manual.mean().item()))

In [ ]:
# 验证非对称性
print('KL(P||Q):', kl_divergence(P, Q).mean().item())
print('KL(Q||P):', kl_divergence(Q, P).mean().item())  # 一般不相等

# JS 散度（对称、有界）
def js_divergence(P, Q, eps=1e-12):
    M = 0.5 * (P + Q)
    return 0.5 * kl_divergence(P, M, eps) + 0.5 * kl_divergence(Q, M, eps)

print('JSD:', js_divergence(P, Q).mean().item(), '(<= log2 =', torch.log(torch.tensor(2.0)).item(), ')')

## 小结 / 易错点
- **输入必须是概率**；拿到 logits 先 softmax。
- `F.kl_div(input, target)` 中 `input` 是 **log Q**、`target` 是 **P**，方向易搞反；默认 `mean` 是按元素平均，求真 KL 要 `batchmean`。
- KL 非对称、非负；要对称用 JS。DPO 损失里用的就是 KL（参考策略与当前策略间）。

## ✅ 测试验证

In [ ]:
# 验证 KL 散度性质
import torch
import torch.nn.functional as F

p = torch.tensor([0.4, 0.6])
q = torch.tensor([0.5, 0.5])

# KL(P||Q) = sum(p * log(p/q))
kl = (p * (p.log() - q.log())).sum()

# 性质1: KL >= 0
assert kl >= 0, f"KL should be non-negative, got {kl}"

# 性质2: KL(P||P) = 0
kl_same = (p * (p.log() - p.log())).sum()
assert abs(kl_same) < 1e-6, f"KL(P||P) should be 0, got {kl_same}"

# 性质3: 与 PyTorch F.kl_div 一致 (注意 F.kl_div 用 q 作为输入，计算 sum(q * (log(q) - log(p))))
# F.kl_div(input=log(q), target=p) = sum(p * (log(p) - log(q))) = KL(P||Q)
kl_torch = F.kl_div(q.log(), p, reduction='sum')
assert torch.allclose(kl, kl_torch, atol=1e-6), f"mismatch: {kl} vs {kl_torch}"

print(f"✅ KLDivergence 测试通过: KL={kl.item():.6f}, 非负且 KL(P||P)=0")
